In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ulianazb/asvspoof-2024/ASVspoof5.train.metadata.txt
/kaggle/input/datasets/ulianazb/asvspoof-2024/ASVspoof5.dev.metadata.txt


In [ ]:
import os
import shutil
import zipfile
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Caminhos das pastas dentro do ambiente do Kaggle
# (O Kaggle monta os datasets dentro do diretório /kaggle/input/)
base_dataset_path = "/kaggle/input/asvspoof-2024"
train_metadata_path = os.path.join(base_dataset_path, "ASVspoof5.train.metadata.txt")
audio_source_folder = os.path.join(base_dataset_path, "train_audio_folder")

# 2. Seu código de metadados que funcionou perfeitamente
parsed_data = []
with open(train_metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            parsed_data.append([parts[0], parts[1], parts[2], parts[3], parts[4]])

columns = ["speaker_id", "audio_id", "gender", "codec", "label"]
df = pd.DataFrame(parsed_data, columns=columns)
df['binary_label'] = df['label'].apply(lambda x: 'bonafide' if x == 'bonafide' else 'spoof')

# 3. Definir amostragem por grupo (250 de cada uma das 4 combinações = 1000 áudios)
samples_per_group = 1000 
balanced_df = df.groupby(['gender', 'binary_label']).apply(
    lambda x: x.sample(n=samples_per_group, random_state=42)
).reset_index(drop=True)

# Separar em treino (80%) e teste (20%)
train_df, test_df = train_test_split(
    balanced_df, 
    test_size=0.20, 
    stratify=balanced_df[['gender', 'binary_label']], 
    random_state=42
)

# Salvar os arquivos CSV de índice que você vai usar no Colab
train_df.to_csv("my_train_split.csv", index=False)
test_df.to_csv("my_test_split.csv", index=False)

# 4. Criar a pasta temporária de saída para os áudios reais
output_audio_folder = "./audios_amostra"
os.makedirs(output_audio_folder, exist_ok=True)

# 5. Copiar cirurgicamente APENAS os 1000 áudios reais que você selecionou
all_required_ids = balanced_df['audio_id'].unique().tolist()
print(f"Copiando {len(all_required_ids)} arquivos de áudio REAIS...")

copiados = 0
for audio_id in all_required_ids:
    nome_arquivo = f"{audio_id}.flac"
    caminho_origem = os.path.join(audio_source_folder, nome_arquivo)
    caminho_destino = os.path.join(output_audio_folder, nome_arquivo)
    
    if os.path.exists(caminho_origem):
        shutil.copy(caminho_origem, caminho_destino)
        copiados += 1
    else:
        # Tenta procurar em subpastas caso a estrutura varie
        for root, dirs, files in os.walk(audio_source_folder):
            if nome_arquivo in files:
                shutil.copy(os.path.join(root, nome_arquivo), caminho_destino)
                copiados += 1
                break

print(f"Total de arquivos reais copiados com sucesso: {copiados}/1000")

# 6. Compactar tudo em um arquivo .zip leve
zip_name = "amostra_asvspoof2024.zip"
print(f"Criando o arquivo compactado {zip_name}...")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_audio_folder):
        for file in files:
            # Salva o arquivo dentro do zip sem carregar toda a estrutura de pastas
            zipf.write(os.path.join(root, file), file)

# Limpar a pasta solta para deixar apenas o arquivo .zip final
shutil.rmtree(output_audio_folder)
print("Processo concluído no Kaggle! O arquivo .zip está pronto para download.")
